## GigaAM from GitHub

### Installing reqs and testing

In [ ]:
! git clone https://github.com/salute-developers/GigaAM.git
%cd GigaAM

Cloning into 'gigaam'...
remote: Enumerating objects: 504, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 504 (delta 3), reused 7 (delta 2), pack-reused 492 (from 1)
Resolving deltas: 100% (298/298), done.
/content/gigaam


In [ ]:
! pip install -e .[tests]

In [5]:
! pytest -v tests/test_loading.py -m partial --disable-warnings

4 passed, 19 deselected in 48.58s


### Model inference

##### Loading arguments

In [8]:
import gigaam

model = gigaam.load_model(
    "v4_rnnt",      # model version: see readme#gigaam-family at GitHub for description
    fp16_encoder=True,  # enabled by default on CUDA, **IGNORED ON CPU (using fp32)**
    use_flash=False,    # disabled by default, boost performance for large tensors (Conformer-only)
    device=None,        # use cuda if available by default, can be set to `cpu` / `cuda`
)

##### Inference methods

In [10]:
# Load test audio
audio_path = gigaam.utils.download_short_audio()

# Audio embeddings
model = gigaam.load_model("v4_ssl")
embedding, _ = model.embed_audio(audio_path)
print("Embeds:", tuple(embedding.shape), embedding.flatten()[:5].tolist())

# ASR
model = gigaam.load_model("v3_ctc")
print("Transcription:", model.transcribe(audio_path))

# Dual-head ASR
model = gigaam.load_model("v4_rnnt")  # also v4_ctc
tr = model.transcribe(audio_path, word_timestamps=True)
print("E2E:", tr["e2e"])
print("Raw:", tr["raw"])
for word in tr["e2e"].words[:5]:
    print(f"  [{word.start:.2f} - {word.end:.2f}] {word.text}")

# Emotion recognition
emotion2prob = gigaam.load_model("emo").get_probs(audio_path)
print("Emotions:", ", ".join(f"{emotion}: {prob:.3f}" for emotion, prob in emotion2prob.items()))

# Voice activity detection
vad = gigaam.load_model("vad")
probs = vad.speech_probs(gigaam.load_audio(audio_path))
print("Speech probabilities:", tuple(probs.shape), probs[:5].tolist())


Embeds: (1, 768, 282) [0.7734375, 0.865234375, 0.6591796875, 0.99072265625, -0.412109375]
Transcription: ничьих не требуя похвал счастлив уж я надеждой сладкой что дева с трепетом любви посмотрит может быть украдкой на песни грешные мои у лукоморья дуб зеленый
E2E: Ничьих не требуя похвал, Счастлив уж я надежды сладкой, Что дева с трепетом любви Посмотрит, может быть, украдкой На песни грешные мои У лукоморья дуб зелёный
Raw: ничьих не требуя похвал счастлив уж я надежды сладкой что дева с трепетом любви посмотрит может быть украдкой на песни грешные мои у лукоморья дуб зеленый
  [0.12 - 0.48] Ничьих
  [0.60 - 0.64] не
  [0.76 - 1.04] требуя
  [1.20 - 1.72] похвал,
  [1.84 - 2.28] Счастлив
Emotions: angry: 0.000, sad: 0.002, neutral: 0.923, positive: 0.074
Speech probabilities: (282,) [0.9999374151229858, 0.9999539852142334, 0.9999560117721558, 0.999953031539917, 0.9999505281448364]


##### Streaming


In [12]:
import random
import torch


def random_audio_pieces(audio, sample_rate=16000, seed=0):
    rng = random.Random(seed)
    offset = 0
    while offset < len(audio):
        size = int(rng.uniform(1.5, 3.0) * sample_rate)
        yield audio[offset:offset + size]
        offset += size


def print_stream(stream, audio):
    utterance = {"raw": "", "e2e": ""}
    for i, piece in enumerate(random_audio_pieces(audio)):
        result = stream.transcribe_partial(piece)
        print(f"Piece {i}:", {head: tr.text for head, tr in result.items()})
        for head, tr in result.items():
            utterance[head] += tr.text
    for head, tr in stream.flush().items():
        utterance[head] += tr.text
    print("Utterance:", utterance)


stream = gigaam.load_model("v4_rnnt", streaming=True, future_context_ms=160)
print_stream(stream, gigaam.load_audio(audio_path))


Piece 0: {'e2e': 'Ничьих не требуя похвал, счастлив уж', 'raw': 'ничьих не требуя похвал счастлив уж'}
Piece 1: {'e2e': ' я надеждой сладкой, Что дева с трепетом любви', 'raw': ' я надеждой сладкой что дева с трепетом любви'}
Piece 2: {'e2e': ' Посмотрит, может быть, украдкой На', 'raw': ' посмотрит может быть украдкой'}
Piece 3: {'e2e': ' песни грешные мои', 'raw': ' на песни грешные мои'}
Piece 4: {'e2e': ' У Лукоморья дуб зелёный', 'raw': ' у лукоморья дуб зеленый'}
Utterance: {'raw': 'ничьих не требуя похвал счастлив уж я надеждой сладкой что дева с трепетом любви посмотрит может быть украдкой на песни грешные мои у лукоморья дуб зеленый', 'e2e': 'Ничьих не требуя похвал, счастлив уж я надеждой сладкой, Что дева с трепетом любви Посмотрит, может быть, украдкой На песни грешные мои У Лукоморья дуб зелёный'}


### ONNX convertation and inference

In [14]:
onnx_dir = "onnx"

gigaam.load_model("v4_ssl").to_onnx(dir_path=onnx_dir)
gigaam.load_model("v4_rnnt").to_onnx(dir_path=onnx_dir)
gigaam.load_model("emo").to_onnx(dir_path=onnx_dir)
gigaam.load_model("vad").to_onnx(dir_path=onnx_dir)


In [15]:
from gigaam.onnx_utils import load_onnx, infer_onnx

sessions, model_cfg = load_onnx(onnx_dir, "v4_ssl")
embedding = infer_onnx(audio_path, model_cfg, sessions, progress=False)[0]
print("Embeds:", embedding.shape, embedding.flatten()[:5].tolist())

sessions, model_cfg = load_onnx(onnx_dir, "v4_rnnt")
tr = infer_onnx(audio_path, model_cfg, sessions, progress=False)
print("E2E:", tr["e2e"][0])
print("Raw:", tr["raw"][0])

sessions, model_cfg = load_onnx(onnx_dir, "emo")
print("Emotions:", infer_onnx(audio_path, model_cfg, sessions, progress=False))
sessions, model_cfg = load_onnx(onnx_dir, "vad")
probs = infer_onnx(audio_path, model_cfg, sessions, progress=False)[0]
print("Speech probabilities:", probs.shape, probs[:5].tolist())


Embeds: (768, 282) [0.774327278137207, 0.8656620383262634, 0.6592346429824829, 0.991009533405304, -0.41168370842933655]
E2E: Ничьих не требуя похвал, Счастлив уж я надежды сладкой, Что дева с трепетом любви Посмотрит, может быть, украдкой На песни грешные мои У лукоморья дуб зелёный
Raw: ничьих не требуя похвал счастлив уж я надежды сладкой что дева с трепетом любви посмотрит может быть украдкой на песни грешные мои у лукоморья дуб зеленый
Emotions: [array([7.7093318e-05, 2.2028075e-03, 9.2327255e-01, 7.4447528e-02],
      dtype=float32)]
Speech probabilities: (282,) [0.9999374151229858, 0.9999539852142334, 0.9999560117721558, 0.999953031539917, 0.9999505281448364]


##### Streaming ONNX


In [17]:
from gigaam.onnx_utils import OnnxGigaAMStreaming

stream = gigaam.load_model("v4_rnnt", streaming=True, future_context_ms=160)
prefix = stream.to_onnx("streaming_onnx")
sessions, cfg = load_onnx("streaming_onnx", prefix)
print_stream(OnnxGigaAMStreaming(cfg, sessions), gigaam.load_audio(audio_path))


Piece 0: {'raw': 'ничьих не требуя похвал счастлив уж', 'e2e': 'Ничьих не требуя похвал, счастлив уж'}
Piece 1: {'raw': ' я надеждой сладкой что дева с трепетом любви', 'e2e': ' я надеждой сладкой, Что дева с трепетом любви'}
Piece 2: {'raw': ' посмотрит может быть украдкой', 'e2e': ' Посмотрит, может быть, украдкой На'}
Piece 3: {'raw': ' на песни грешные мои', 'e2e': ' песни грешные мои'}
Piece 4: {'raw': ' у лукоморья дуб зеленый', 'e2e': ' У Лукоморья дуб зелёный'}
Utterance: {'raw': 'ничьих не требуя похвал счастлив уж я надеждой сладкой что дева с трепетом любви посмотрит может быть украдкой на песни грешные мои у лукоморья дуб зеленый', 'e2e': 'Ничьих не требуя похвал, счастлив уж я надеждой сладкой, Что дева с трепетом любви Посмотрит, может быть, украдкой На песни грешные мои У Лукоморья дуб зелёный'}


### Longform

Use `.transcribe_longform` for long audio. The public Conformer VAD checkpoint is downloaded automatically.

In [19]:
long_audio_path = gigaam.utils.download_long_audio()
model = gigaam.load_model("v4_rnnt")
result = model.transcribe_longform(long_audio_path)
for segment in result["e2e"]:
    print(f"[{gigaam.format_time(segment.start)} - {gigaam.format_time(segment.end)}]: {segment.text}")


[00:00:00 - 00:29:70]: Вечерня отошла давно, Но в кельях тихо и темно Уже и сам игумен строгий Свои молитвы прекратил И кости ветхие склонил Перекрестясь На одр убогий Кругом и сон и тишина Но церкви дверь отворена Трепещет луч лампады И тускло озаряет он И тёмну живопись икон, И позлащённые оклады И раздаётся в тишине То тяжкий вздох, то шёпот важный
[00:29:70 - 00:39:40]: И мрачно дремлет в вышине Старинный свод Глухой и влажный Стоят за клиросом Чернец и грешник Неподвижны оба
[00:39:71 - 00:46:20]: И шёпот их как глас из гроба, И грешник бледен, как мертвец монах.
[00:46:75 - 01:11:15]: Несчастный, полно перестань: Ужасна исповедь злодея Заплачена тобою дань Тому, кто в злобе пламенея Лукаво грешника блюдёт И к вечной гибели ведёт Смирись, опомнись время, время раскаянья покров Я разрешу тебя Грехов сложи Мучительное бремя.


### More advanced examples

##### Another way to load audio

In [22]:
import librosa
import torch

from gigaam.utils import AudioDataset
from gigaam.preprocess import SAMPLE_RATE as SR

fname = gigaam.utils.download_short_audio()
wav = gigaam.load_audio(fname)
wav_ = torch.from_numpy(librosa.load(fname, sr=SR, mono=True)[0])
wav_tns, lengths = AudioDataset.collate([wav, wav_])
with torch.inference_mode():
    encoded, encoded_len = model(wav_tns.to(model._device), lengths.to(model._device))
    for texts in model.decode(encoded, encoded_len).values():
        assert texts[0] == texts[1]


Compare batched and unbatched inference on the same independent longform segments.


In [24]:
import time
from gigaam.vad_utils import segment_audio_file

long_fname = gigaam.utils.download_long_audio()
model = gigaam.load_model("v4_rnnt")
segments, _ = segment_audio_file(long_fname, sr=SR, device=model._device)


def inference_seconds(batch_size):
    loader = torch.utils.data.DataLoader(
        segments, batch_size=batch_size, collate_fn=AudioDataset.collate, shuffle=False
    )
    if model._device.type == "cuda":
        torch.cuda.synchronize(model._device)
    start = time.perf_counter()
    with torch.inference_mode():
        for wav_tns, lengths in loader:
            encoded, encoded_len = model(wav_tns.to(model._device), lengths.to(model._device))
            model.decode(encoded, encoded_len)
    if model._device.type == "cuda":
        torch.cuda.synchronize(model._device)
    return time.perf_counter() - start


print(f"Batched: {inference_seconds(4):.3f} seconds")
print(f"Unbatched: {inference_seconds(1):.3f} seconds")


Batched: 1.033 seconds
Unbatched: 3.674 seconds


## GigaAM from Hugging Face

### Reqs and files loading: now by hand

In [ ]:
! pip install numpy "transformers<5" onnx onnxruntime hydra-core omegaconf sentencepiece tqdm


In [ ]:
! wget https://cdn.chatwm.opensmodel.sberdevices.ru/GigaAM/example.wav
! wget https://cdn.chatwm.opensmodel.sberdevices.ru/GigaAM/long_example.wav

In [29]:
from transformers import AutoModel

repo_name = "ai-sage/GigaAM-v4"
model = AutoModel.from_pretrained(repo_name, revision="rnnt", trust_remote_code=True)


### Everything else is the same!

> Note: `fp16` encoder on CUDA can be used with `model.model.encoder = model.model.encoder.half()`.

In [31]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModel.from_pretrained(repo_name, revision="ssl", trust_remote_code=True).to(device)
embedding, _ = model.embed_audio("example.wav")
print("Embed:", tuple(embedding.shape), embedding.flatten()[:5].tolist())
model = AutoModel.from_pretrained(repo_name, revision="rnnt", trust_remote_code=True).to(device)
tr = model.transcribe("example.wav")
print("E2E:", tr["e2e"])
print("Raw:", tr["raw"])
print_stream(model.create_stream(future_context_ms=160), model.model.prepare_wav("example.wav")[0][0])


Embed: (1, 768, 282) [0.7737169861793518, 0.8643113970756531, 0.6599801778793335, 0.9907341003417969, -0.4116284251213074]
E2E: Ничьих не требуя похвал, Счастлив уж я надежды сладкой, Что дева с трепетом любви Посмотрит, может быть, украдкой На песни грешные мои У лукоморья дуб зелёный
Raw: ничьих не требуя похвал счастлив уж я надежды сладкой что дева с трепетом любви посмотрит может быть украдкой на песни грешные мои у лукоморья дуб зеленый
Piece 0: {'e2e': 'Ничьих не требуя похвал, счастлив уж', 'raw': 'ничьих не требуя похвал счастлив уж'}
Piece 1: {'e2e': ' я надеждой сладкой, Что дева с трепетом любви', 'raw': ' я надеждой сладкой что дева с трепетом любви'}
Piece 2: {'e2e': ' Посмотрит, может быть, украдкой На', 'raw': ' посмотрит может быть украдкой'}
Piece 3: {'e2e': ' песни грешные мои', 'raw': ' на песни грешные мои'}
Piece 4: {'e2e': ' У Лукоморья дуб зелёный', 'raw': ' у лукоморья дуб зеленый'}
Utterance: {'raw': 'ничьих не требуя похвал счастлив уж я надеждой сладкой что 

In [32]:
model.to_onnx("hf_onnx")
sessions, cfg = model.load_onnx("hf_onnx", model.model.cfg.model_name)
tr = model.infer_onnx("example.wav", cfg, sessions, progress=False)
print("E2E:", tr["e2e"][0])
print("Raw:", tr["raw"][0])

prefix = model.to_onnx("hf_streaming_onnx", streaming=True)
sessions, cfg = model.load_onnx("hf_streaming_onnx", prefix)
print_stream(model.create_onnx_stream(cfg, sessions), model.model.prepare_wav("example.wav")[0][0])


E2E: Ничьих не требуя похвал, Счастлив уж я надежды сладкой, Что дева с трепетом любви Посмотрит, может быть, украдкой На песни грешные мои У лукоморья дуб зелёный
Raw: ничьих не требуя похвал счастлив уж я надежды сладкой что дева с трепетом любви посмотрит может быть украдкой на песни грешные мои у лукоморья дуб зеленый
Piece 0: {'e2e': 'Ничьих, не требуя похвал, счастлив у', 'raw': 'ничьих не требуя похвал счастлив у'}
Piece 1: {'e2e': 'ж я надеждой сладкой, Что дева с трепетом люб', 'raw': 'ж я надежды сладкой что дева с трепетом люб'}
Piece 2: {'e2e': 'ви Посмотрит может быть укра', 'raw': 'ви посмотрит может быть укра'}
Piece 3: {'e2e': 'дкой На песни грешные мои', 'raw': 'дкой на песни грешные мои'}
Piece 4: {'e2e': ' У Лукоморья дуб зелё', 'raw': ' у лукоморья дуб зелен'}
Utterance: {'raw': 'ничьих не требуя похвал счастлив уж я надежды сладкой что дева с трепетом любви посмотрит может быть украдкой на песни грешные мои у лукоморья дуб зеленый', 'e2e': 'Ничьих, не требуя похвал

### GigaAM-Multilingual

Same interface, different repo. Revisions: `ssl`, `rnnt`.


In [34]:
model = AutoModel.from_pretrained(
    "ai-sage/GigaAM-Multilingual-v2", revision="rnnt",
    trust_remote_code=True, language="ru",
).to(device)
tr = model.transcribe("example.wav", word_timestamps=True)
print("E2E:", tr["e2e"])
print("Raw:", tr["raw"])
for word in tr["e2e"].words[:5]:
    print(word)
print_stream(model.create_stream(future_context_ms=160), model.model.prepare_wav("example.wav")[0][0])


E2E: Ничьих не требуя похвал, Счастлив уж я надеждой сладкой, Что дева с трепетом любви Посмотрит, может быть, украдкой На песни грешные мои У лукоморья дуб зелёный
Raw: ничьих не требуя похвал счастлив уж я надеждой сладкой что дева с трепетом любви посмотрит может быть украдкой на песни грешные мои у лукоморья дуб зелёный
Word(text='Ничьих', start=0.1601418439716312, end=0.5604964539007092)
Word(text='не', start=0.6405673758865248, end=0.7606737588652482)
Word(text='требуя', start=0.7606737588652482, end=1.1209929078014185)
Word(text='похвал,', start=1.201063829787234, end=1.7615602836879432)
Word(text='Счастлив', start=1.7215248226950355, end=2.1218794326241133)
Piece 0: {'e2e': 'Ничьих не требуя похвал, счастлив уж', 'raw': 'ничьих не требуя похвал счастлив у'}
Piece 1: {'e2e': ' я надеждой сладкой, Что дева с трепетом любви', 'raw': 'ж я надеждой сладкой что дева с трепетом люб'}
Piece 2: {'e2e': ' Посмотрит, может быть, украдкой На', 'raw': 'ви посмотрит может быть украдкой на'}
